# Stage 4：Olist 訂單是否延遲預測

目標：使用下單當下可取得的訂單、付款、商品、顧客與賣家資訊，預測訂單是否晚於預估日期送達。

`delayed = 1`：實際送達日期晚於預估送達日期；否則為 0。

為避免資料洩漏，實際配送天數、承運日期、實際送達日期不會放入模型特徵。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_olist_delay_model import (
    load_and_build_features,
    temporal_train_test_split,
    evaluate_models,
    get_feature_importance,
    build_risk_segments,
)

print(PROJECT_ROOT)

## 1. 建立模型資料集

每列代表一筆已送達訂單。商品與付款明細會先聚合到訂單層級，再與顧客、賣家、商品資料合併。

In [ ]:
dataset, numeric_features, categorical_features = load_and_build_features()
print(f"可用訂單：{len(dataset):,}")
print(f"延遲率：{dataset['delayed'].mean():.2%}")
dataset.head()

## 2. 時間切分

用較早的 80% 訂單訓練、較新的 20% 訂單測試，比隨機切分更接近實際上線後預測未來訂單的情境。

In [ ]:
train, test = temporal_train_test_split(dataset)
print("Train:", train.shape, train['order_purchase_timestamp'].min(), "→", train['order_purchase_timestamp'].max())
print("Test: ", test.shape, test['order_purchase_timestamp'].min(), "→", test['order_purchase_timestamp'].max())
print("Train delay rate:", f"{train['delayed'].mean():.2%}")
print("Test delay rate: ", f"{test['delayed'].mean():.2%}")

## 3. 模型比較

比較 Logistic Regression、Decision Tree 與 Random Forest。因延遲訂單比例較低，模型使用 class weight，評估時同時觀察 Precision、Recall、F1、ROC-AUC 與 Average Precision。

In [ ]:
metrics, models, probabilities = evaluate_models(
    train, test, numeric_features, categorical_features
)
metrics

## 4. 最佳模型與重要特徵

In [ ]:
best_model_name = metrics.iloc[0]['model']
best_model = models[best_model_name]
feature_importance = get_feature_importance(best_model)
print("Best model:", best_model_name)
feature_importance.head(20)

## 5. 哪類訂單風險最高？

以下只使用時間切分後的測試集，依商品品類、顧客州別、賣家州別與下單月份彙整；為降低小樣本波動，只保留至少 100 筆訂單的群組。

In [ ]:
risk_segments = build_risk_segments(test, probabilities[best_model_name])
risk_segments.head(20)

## 6. 完整產出

執行 `python scripts/train_olist_delay_model.py` 會儲存：

- 最佳模型與模型 metadata
- 模型比較、特徵重要性、風險分群與測試集預測 CSV
- 混淆矩陣、ROC、PR curve、模型比較與特徵重要性圖
- `reports/stage4_delay_prediction_summary.md` 結果摘要